In [ ]:
## READS ON POSTPROCESS pqt, saves to disk
## Holy grail theory
## v1 - HA + JMA
# wish list:
# one more nudge/signal - whether HA candle body crosses JMA (need to write list of 
# handcrafted wick/body signals - try GBM first see if they add something statistically

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [ ]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [ ]:
%%time
file_name = 'ha-2sec-jma'
res_file = f'/mnt/c/ml/data/2025/holy-grail/{file_name}.csv'
df = pd.read_csv(res_file)

In [ ]:

# schema
type_spec = {
    #'Open': 'float32',
    #'High': 'float32',
    #'Low': 'float32',
    #'Close': 'float32',
    #'haBody': 'float32',
    #'haWickTop': 'float32',
    #'haWickBott': 'float32',
    'haColour': 'int8',
    #'jma': 'float32',
    'slopeSignJma': 'int8',
    'segmentId': 'int16',
    'segmentLength': 'int8',
    'G': 'int8',
    'g': 'int8',
}

# Apply conversion
df = df.astype(type_spec)

# cast to datetime
df['DateTime'] = pd.to_datetime(df['DateTime'])

print(df.info())

In [ ]:
%%time

if not df["DateTime"].is_monotonic_increasing:
    raise ValueError("Unexpected unordered timestamp rows found")

df['date'] = df['DateTime'].dt.normalize()

day = pd.Timestamp("2025-11-28")
df = df[df["date"] != day]

In [ ]:
print(df["segmentLength"].min())

In [ ]:
%%time

"""
S2 input builder. Reads raw C# feature parquet, adds engineered ratio/trajectory
features, writes two model-input versions: 'signed' (direction intact) and
'symmetric' (direction folded out via slopeSign). Run both through the GBM probe
to test whether sign carries the edge.

Output is the versioned artifact you commit to git alongside the model version.
"""

#RAW = "features_raw.parquet"          # <-- raw C# output
OUT_SIGNED = f"{file_name}-signed-v2.pqt"
EPS = 1e-9

# ---- integrity (the checks we already validated) ----
#df = df[df["timestamp"].dt.date != pd.Timestamp("2025-11-28").date()].copy()
df["remaining"] = df["segmentLength"] - df["g"]
assert df["segmentLength"].min() == 1, "zero-length segment present"
assert df["remaining"].min() == 0, "g overruns segmentLength"

date = df['date']

# ---- sign-agnostic engineered features ----
df["wickTop_body"] = df["haWickTop"] / (df["haBody"] + EPS)
df["wickBot_body"] = df["haWickBott"] / (df["haBody"] + EPS)
df["wickAsym"]     = (df["haWickBott"] - df["haWickTop"]) / (df["haBody"] + EPS)
df["wickSum_body"] = (df["haWickTop"] + df["haWickBott"]) / (df["haBody"] + EPS)
df["bodyRange"]    = df["haBody"] / (df["haBody"] + df["haWickTop"] + df["haWickBott"] + EPS)

# ---- short causal trajectory deltas (within-day; NaN at each day's first k rows) ----
gd = df.groupby(date, sort=False)
df["dBody_3"]     = gd["haBody"].diff(3).fillna(0.0).values
df["dWickTopR_3"] = gd["wickTop_body"].diff(3).fillna(0.0).values
df["dWickBotR_3"] = gd["wickBot_body"].diff(3).fillna(0.0).values

# trim columns
signed = df.drop(columns=[
    'Open',
    'High',
    'Low',
    'Close',
    'jma',
    'segmentLength',
    'G',])

signed.rename(columns={'DateTime': 'timestamp'},inplace=True)

# write
signed.to_parquet(OUT_SIGNED, index=False)
print(f"wrote {OUT_SIGNED} rows={len(df):,} days={date.nunique()}")

#build(True).to_parquet(OUT_SYMMETRIC, index=False)
#print(f"wrote {OUT_SIGNED} and {OUT_SYMMETRIC}  rows={len(df):,}  days={date.nunique()}")


In [ ]:
print(signed.info())
print(signed.head())
print(signed.tail())

In [ ]:
x = 1/0 ## the rest is useless - score removed, fields renamed

print(
    df['segmentLength']
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_string(float_format=lambda x: f"{x:.1f}")
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(20, 5))
plt.hist(df['segmentLength'], bins=200, color='steelblue', edgecolor='black')
plt.xlabel("segmentLength")
plt.ylabel("Count")
plt.title("Distribution of segmentLength")
plt.tight_layout()
plt.show()


In [ ]:
# segmentId resets to 0 each day (per-call), so it is NOT unique across the concat.
# Build a real per-segment id from value-changes:
df['seg_uid'] = (df['segmentId'] != df['segmentId'].shift()).cumsum()

seg = df.groupby('seg_uid')['segmentLength'].first()
print(seg.describe(percentiles=[.5, .75, .9, .95, .99]))
print('frac <=3 :', (seg <= 3).mean())
print('frac ==1 :', (seg == 1).mean())

# label-machinery sanity (must hold while factor==1):
slen = df['segmentLength'].astype('float64')
print('score_g==g :', (df['score_g'] == df['g']).all())
print('score==tri :', np.allclose(df['score'], slen * (slen + 1) / 2))
print('nan        :', df[['d1','d2','haBody','haWickTop','haWickBott']].isna().any().any())

In [ ]:
z = df[df['segmentLength'] == 0]
print(len(z), z['seg_uid'].nunique())
print(z.groupby('seg_uid').size().describe())     # how big are the zero-len groups
print((z['timestamp'].dt.date.value_counts()).head())  # concentrated in certain days?

In [ ]:
df['remaining'] = df['segmentLength'] - df['g']
print('min remain :', df['remaining'].min())   # MUST be 0; <0 = g exceeds its segment length (C# bug)
print('rows       :', len(df))
print('days       :', df['timestamp'].dt.date.nunique())